<a href="https://colab.research.google.com/github/OJB-Quantum/Notebooks-for-Ideas/blob/main/IBM_Granite4_small_in_colab_demo_uv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Authored by Onri Jay Benally (2026)

Open Access (CC-BY-4.0)

# Deploying International Business Machines (IBM) Granite 4 Small via Google Colaboratory (Colab)

## Primer
This document outlines the procedural workflow for deploying the IBM Granite 4 Small language model within a Google Colaboratory environment. To optimize installation speeds and model serving, we utilize `uv` for Python dependency management and Ollama as the local inference server. This architecture is designed specifically for high-performance compute nodes tailored to handle extensive memory loads and long-context operations.

## Acronyms and Definitions
| Acronym/ Term | Definition |
| :--- | :--- |
| **Colab** | Google Colaboratory; a cloud-based Jupyter notebook environment. |
| **GB** | Gigabyte; a unit of digital information storage capacity. |
| **GPU** | Graphics Processing Unit; specialized hardware for parallel processing tasks. |
| **IBM** | International Business Machines Corporation. |
| **VRAM** | Video Random Access Memory; dedicated memory utilized by the GPU. |
| **`uv`** | A high-performance Python package installer and resolver. |

## Workflow Overview

This execution sequence performs the following operations:

1. **Hardware Verification:** Detects the active Graphics Processing Unit (GPU) and outputs a capacity status to ensure sufficient Video Random Access Memory (VRAM) availability.
2. **Environment Initialization:** Installs required Linux system packages, the `uv` package manager, and the Ollama daemon.
3. **Service Activation:** Initializes the `ollama serve` background process within the Colab runtime.
4. **Model Acquisition:** Downloads the `ibm/granite4:small-h` weights from the model registry.
5. **Client Configuration:** Installs the Ollama Python client leveraging `uv` for rapid dependency resolution.
6. **Validation Testing:** Executes a baseline system prompt and provisions an interactive conversational interface.

## Infrastructure Requirements

- **Compute Target:** This architecture strictly requires a runtime equipped with a hardware accelerator (GPU).
- **Memory Allocation:** The Granite 4 Small architecture demands extensive memory for long-context operations. A **G4 class instance** (featuring 96 GB of VRAM) represents the optimal deployment target to ensure stability and maximize the theoretical context window.
- **Storage Overhead:** The initial model acquisition phase consumes significant network bandwidth and local disk storage. Allocate sufficient time for this preliminary download prior to execution.

## Control knobs

Adjust these values before running the install and inference cells.

| Name | Meaning |
|---|---|
| `model_name` | Ollama model tag to pull and run |
| `ollama_host` | Host interface for the Ollama server |
| `ollama_port` | TCP port for the Ollama server |
| `models_dir` | Directory for downloaded Ollama model files |
| `log_path` | Log file for `ollama serve` |
| `cuda_visible_devices` | GPU device selection string |
| `num_ctx` | Requested context window size for chat calls |
| `uv_bin_dir` | Install location for the `uv` binary |
| `install_uv` | Whether to install `uv` |
| `install_ollama` | Whether to install Ollama |


In [10]:
from __future__ import annotations

import json
import os
import socket
import subprocess
import time
import urllib.error
import urllib.request
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional


@dataclass(frozen=True)
class ColabOllamaConfig:
    """Configuration for installing and running Ollama in Colab."""

    model_name: str = "ibm/granite4:small-h"
    ollama_host: str = "127.0.0.1"
    ollama_port: int = 11434
    models_dir: Path = Path("/content/ollama_models")
    log_path: Path = Path("/content/ollama_serve.log")
    cuda_visible_devices: str = "0"
    num_ctx: int = 131072
    uv_bin_dir: Path = Path("/content/.local/bin")
    install_uv: bool = True
    install_ollama: bool = True


CFG = ColabOllamaConfig()

In [11]:
def _is_root() -> bool:
    """Return True if the current process has root privileges."""
    try:
        return os.geteuid() == 0
    except AttributeError:
        return False


def _sudo_prefix() -> str:
    """Return 'sudo ' when needed, otherwise ''."""
    return "" if _is_root() else "sudo "


def run_bash(command: str, *, check: bool = True) -> subprocess.CompletedProcess:
    """Run a bash command in a Colab friendly way."""
    print(f"\n[run] {command}\n")
    return subprocess.run(["bash", "-lc", command], check=check)


def capture_bash(command: str) -> str:
    """Run a bash command and capture stdout as text."""
    out = subprocess.check_output(["bash", "-lc", command], text=True)
    return out.strip()


def prepend_to_path(path: Path) -> None:
    """Prepend a directory to PATH for subsequent subprocess calls."""
    path_str = str(path)
    path_parts = os.environ.get("PATH", "").split(":")
    if path_str not in path_parts:
        os.environ["PATH"] = f"{path_str}:{os.environ.get('PATH', '')}"


def is_tcp_port_open(host: str, port: int, timeout_s: float = 0.25) -> bool:
    """Return True if a TCP port is accepting connections."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout_s)
        return sock.connect_ex((host, port)) == 0


def wait_for_ollama_ready(
    host: str,
    port: int,
    timeout_s: float = 30.0,
    poll_s: float = 0.5,
) -> dict:
    """Wait until the Ollama server responds to GET /api/version."""
    url = f"http://{host}:{port}/api/version"
    deadline = time.time() + timeout_s
    last_err: Optional[Exception] = None

    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=2.0) as resp:
                payload = resp.read().decode("utf-8")
            return json.loads(payload)
        except (urllib.error.URLError, json.JSONDecodeError) as err:
            last_err = err
            time.sleep(poll_s)

    raise TimeoutError(f"Ollama did not become ready: {last_err}")


def nvidia_smi_summary() -> str:
    """Return a concise GPU summary from nvidia-smi, if available."""
    command = "command -v nvidia-smi >/dev/null 2>&1"
    if subprocess.call(["bash", "-lc", command]) != 0:
        return "nvidia-smi not found (GPU runtime may be disabled)."

    query = (
        "nvidia-smi --query-gpu=name,driver_version,memory.total "
        "--format=csv,noheader"
    )
    return capture_bash(query)


def first_gpu_total_memory_gib(gpu_summary: str) -> Optional[float]:
    """Parse the first GPU total memory, in GiB, from nvidia-smi summary text."""
    if not gpu_summary or "MiB" not in gpu_summary:
        return None

    first_line = gpu_summary.splitlines()[0]
    parts = [part.strip() for part in first_line.split(",")]
    if len(parts) < 3:
        return None

    memory_text = parts[2].upper().replace("MIB", "").strip()
    try:
        return float(memory_text) / 1024.0
    except ValueError:
        return None


def warn_for_granite4_small_capacity(gpu_summary: str) -> None:
    """Print a Granite 4 Small capacity warning based on visible GPU details."""
    memory_gib = first_gpu_total_memory_gib(gpu_summary)
    upper = gpu_summary.upper()

    if "A100" in upper and memory_gib is not None and memory_gib >= 39.0:
        print(
            "[gpu] Detected an A100 class runtime. This is the safer Colab target "
            "for Granite 4 Small."
        )
        return

    if "L4" in upper:
        print(
            "[gpu] WARNING: Detected an L4 class runtime. Granite 4 Small is large, "
            "so reduce `CFG.num_ctx` if you encounter out of memory errors, and keep "
            "other GPU memory use minimal."
        )
        return

    print(
        "[gpu] WARNING: Granite 4 Small is a relatively large model. An A100 class "
        f"GPU is the safer target. Detected runtime: {gpu_summary}"
    )

## Runtime sanity checks

Run this first to confirm that the notebook is attached to a GPU runtime and that `/content` has adequate free disk space.

In [12]:
gpu = nvidia_smi_summary()
print(gpu)
warn_for_granite4_small_capacity(gpu)

run_bash("df -h /content")
run_bash("uname -a")

NVIDIA RTX PRO 6000 Blackwell Server Edition, 580.82.07, 97887 MiB
[gpu] WARNING: Granite 4 Small is a relatively large model. An A100 class GPU is the safer target. Detected runtime: NVIDIA RTX PRO 6000 Blackwell Server Edition, 580.82.07, 97887 MiB

[run] df -h /content


[run] uname -a



CompletedProcess(args=['bash', '-lc', 'uname -a'], returncode=0)

## Install baseline packages, `uv`, and Ollama

This cell uses:

- `apt-get` for required Linux tools,
- Astral's `uv` installer in unmanaged mode,
- Ollama's official Linux installer.

Then it verifies that both `uv` and `ollama` are available.

In [13]:
sudo = _sudo_prefix()

if CFG.install_ollama or CFG.install_uv:
    run_bash(f"{sudo}apt-get update -y")
    run_bash(
        f"{sudo}apt-get install -y "
        "curl ca-certificates zstd pciutils lshw"
    )

if CFG.install_uv:
    CFG.uv_bin_dir.mkdir(parents=True, exist_ok=True)
    run_bash(
        "curl -LsSf https://astral.sh/uv/install.sh | "
        f'env UV_UNMANAGED_INSTALL="{CFG.uv_bin_dir}" sh'
    )

prepend_to_path(CFG.uv_bin_dir)
run_bash("command -v uv")
run_bash("uv --version")

if CFG.install_ollama:
    run_bash("curl -fsSL https://ollama.com/install.sh | sh")

run_bash("command -v ollama")
run_bash("ollama -v")


[run] apt-get update -y


[run] apt-get install -y curl ca-certificates zstd pciutils lshw


[run] curl -LsSf https://astral.sh/uv/install.sh | env UV_UNMANAGED_INSTALL="/content/.local/bin" sh


[run] command -v uv


[run] uv --version


[run] curl -fsSL https://ollama.com/install.sh | sh


[run] command -v ollama


[run] ollama -v



CompletedProcess(args=['bash', '-lc', 'ollama -v'], returncode=0)

## Start `ollama serve` and pull the Granite 4 Small model

This cell starts the local Ollama server, waits for the version endpoint to become ready, and then pulls the model.

In [14]:
CFG.models_dir.mkdir(parents=True, exist_ok=True)

if is_tcp_port_open(CFG.ollama_host, CFG.ollama_port):
    print(
        f"[ollama] Server already listening on "
        f"{CFG.ollama_host}:{CFG.ollama_port}"
    )
else:
    env = os.environ.copy()
    env["OLLAMA_HOST"] = f"{CFG.ollama_host}:{CFG.ollama_port}"
    env["OLLAMA_MODELS"] = str(CFG.models_dir)
    env["CUDA_VISIBLE_DEVICES"] = CFG.cuda_visible_devices

    CFG.log_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"[ollama] Logging to: {CFG.log_path}")

    with CFG.log_path.open("a", encoding="utf-8") as log_file:
        proc = subprocess.Popen(
            ["ollama", "serve"],
            env=env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
        )

    print(f"[ollama] Started server PID={proc.pid}")

version_payload = wait_for_ollama_ready(CFG.ollama_host, CFG.ollama_port)
print("[ollama] /api/version =>", version_payload)

run_bash(f"ollama pull {CFG.model_name}")

[ollama] Server already listening on 127.0.0.1:11434
[ollama] /api/version => {'version': '0.18.2'}

[run] ollama pull ibm/granite4:small-h



CompletedProcess(args=['bash', '-lc', 'ollama pull ibm/granite4:small-h'], returncode=0)

## Install the Ollama Python client with `uv` and run a smoke test

This uses `uv pip install --system` so the package lands in the active Colab Python environment.

In [15]:
run_bash("uv pip install --system --upgrade ollama")
run_bash("ollama list")


[run] uv pip install --system --upgrade ollama


[run] ollama list



CompletedProcess(args=['bash', '-lc', 'ollama list'], returncode=0)

In [16]:
from ollama import chat

response = chat(
    model=CFG.model_name,
    messages=[{"role": "user", "content": "Hello!"}],
    options={"num_ctx": CFG.num_ctx},
)
print(response.message.content)

run_bash("ollama ps")
run_bash("nvidia-smi")

Hello! How may I assist you today?

[run] ollama ps


[run] nvidia-smi



CompletedProcess(args=['bash', '-lc', 'nvidia-smi'], returncode=0)

## Interactive chat loop

Run this cell after the model is installed. Type `quit`, `exit`, or `q` to stop the loop.

If you hit memory pressure, reduce `CFG.num_ctx` in the configuration cell and rerun the relevant cells.

In [17]:
from ollama import chat

MODEL_NAME = CFG.model_name
NUM_CTX = CFG.num_ctx
SYSTEM_PROMPT = ""

messages: List[Dict[str, str]] = []
if SYSTEM_PROMPT.strip():
    messages.append({"role": "system", "content": SYSTEM_PROMPT.strip()})


def send_turn(user_text: str) -> str:
    """Send one user turn and return one assistant turn."""
    messages.append({"role": "user", "content": user_text})
    response = chat(
        model=MODEL_NAME,
        messages=messages,
        options={"num_ctx": NUM_CTX},
    )
    assistant_text = response.message.content
    messages.append({"role": "assistant", "content": assistant_text})
    return assistant_text


while True:
    user_text = input("\nYou: ").strip()
    if user_text.lower() in {"quit", "exit", "q"}:
        print("Stopped.")
        break
    if not user_text:
        continue

    try:
        reply = send_turn(user_text)
    except Exception as exc:
        raise RuntimeError(
            "Prompting failed. Verify these in earlier cells:\n"
            "  1) `ollama serve` is running and reachable on localhost:11434\n"
            "  2) the model is present "
            "(`ollama pull ibm/granite4:small-h`)\n"
            "  3) the `ollama` Python package is installed "
            "(`uv pip install --system ollama`)\n"
            "  4) `NUM_CTX` is not too large for the available GPU memory\n"
        ) from exc

    print(f"\nAssistant: {reply}")


You: q
Stopped.


## Operational notes

- Reconnecting to a new Colab runtime means the local server and installed binaries may need to be recreated.
- The model files are stored under `CFG.models_dir`, which defaults to `/content/ollama_models`.
- For tighter GPU memory budgets, start by lowering `CFG.num_ctx`.
